# Real-Time Grid Monitoring & Anomaly Detection - Synthetic Data Generator

## Overview
This notebook generates synthetic data simulating a power grid monitoring scenario with:
- **SCADA measurements** - voltage, current, power, frequency
- **PMU (Phasor Measurement Unit) data** - high-frequency synchrophasor readings
- **Weather data** - temperature, humidity affecting grid performance
- **Anomaly events** - realistic grid disturbances and faults

## Tables Created
| Table | Description | Volume |
|-------|-------------|--------|
| `dim_substations` | Substation reference data | ~50 rows |
| `dim_equipment` | Equipment (transformers, breakers) | ~200 rows |
| `fact_scada_measurements` | SCADA telemetry | 1M+ rows/day |
| `fact_pmu_readings` | PMU synchrophasor data | 10M+ rows/day |
| `fact_weather` | Weather conditions | 1K rows/day |
| `fact_anomaly_events` | Detected anomalies | ~100 rows/day |

## How to Run in Fabric
1. Attach this notebook to a Lakehouse
2. Run all cells sequentially
3. Data will be written as Delta tables to the attached Lakehouse
4. For streaming simulation, schedule this notebook to run every minute

In [ ]:
# Install required packages
!pip install faker

In [ ]:
# Configuration Parameters
SEED = 42
NUM_SUBSTATIONS = 50
NUM_EQUIPMENT_PER_SUBSTATION = 4
SCADA_READINGS_PER_MINUTE = 100
PMU_READINGS_PER_SECOND = 30
ANOMALY_PROBABILITY = 0.02  # 2% chance of anomaly per reading

# Date range for historical data
from datetime import datetime, timedelta
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 1, 7)  # 7 days of historical data

# Streaming mode settings
STREAMING_MODE = False  # Set to True for incremental generation
STREAMING_INTERVAL_SECONDS = 60

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

# Set random seeds for reproducibility
np.random.seed(SEED)
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

print("Libraries loaded successfully")

## 1. Generate Dimension Tables

In [ ]:
# Generate Substations
def generate_substations(n):
    voltage_levels = [69, 115, 138, 230, 345, 500]  # kV
    regions = ['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West']
    
    substations = []
    for i in range(n):
        substations.append({
            'substation_id': f'SUB-{str(i+1).zfill(4)}',
            'substation_name': f'{fake.city()} Substation',
            'voltage_level_kv': random.choice(voltage_levels),
            'region': random.choice(regions),
            'latitude': round(random.uniform(25, 48), 6),
            'longitude': round(random.uniform(-125, -70), 6),
            'commissioned_date': fake.date_between(start_date='-30y', end_date='-1y'),
            'is_critical': random.random() < 0.2
        })
    return pd.DataFrame(substations)

df_substations = generate_substations(NUM_SUBSTATIONS)
print(f"Generated {len(df_substations)} substations")
df_substations.head()

In [ ]:
# Generate Equipment
def generate_equipment(substations_df, equipment_per_sub):
    equipment_types = [
        ('Transformer', 'XFMR'),
        ('Circuit Breaker', 'CB'),
        ('Capacitor Bank', 'CAP'),
        ('Current Transformer', 'CT')
    ]
    manufacturers = ['ABB', 'Siemens', 'GE', 'Schneider', 'Hitachi']
    
    equipment = []
    for _, sub in substations_df.iterrows():
        for i in range(equipment_per_sub):
            eq_type, eq_code = random.choice(equipment_types)
            equipment.append({
                'equipment_id': f'{sub["substation_id"]}-{eq_code}-{str(i+1).zfill(2)}',
                'substation_id': sub['substation_id'],
                'equipment_type': eq_type,
                'manufacturer': random.choice(manufacturers),
                'model': f'{fake.lexify(text="???").upper()}-{random.randint(100,999)}',
                'install_date': fake.date_between(start_date='-20y', end_date='-6m'),
                'rated_voltage_kv': sub['voltage_level_kv'],
                'rated_current_a': random.choice([600, 1200, 2000, 3000])
            })
    return pd.DataFrame(equipment)

df_equipment = generate_equipment(df_substations, NUM_EQUIPMENT_PER_SUBSTATION)
print(f"Generated {len(df_equipment)} equipment records")
df_equipment.head()

## 2. Generate SCADA Measurements

In [ ]:
def generate_scada_readings(equipment_df, start_time, end_time, readings_per_minute):
    """Generate SCADA measurements with realistic patterns and occasional anomalies"""
    
    readings = []
    current_time = start_time
    interval = timedelta(seconds=60 / readings_per_minute)
    
    # Base values for different equipment types
    base_values = {
        'Transformer': {'voltage': 1.0, 'current': 0.7, 'power': 0.75, 'frequency': 60.0},
        'Circuit Breaker': {'voltage': 1.0, 'current': 0.5, 'power': 0.5, 'frequency': 60.0},
        'Capacitor Bank': {'voltage': 1.02, 'current': 0.3, 'power': 0.2, 'frequency': 60.0},
        'Current Transformer': {'voltage': 1.0, 'current': 0.6, 'power': 0.4, 'frequency': 60.0}
    }
    
    while current_time < end_time:
        # Add daily load pattern (higher during day, lower at night)
        hour = current_time.hour
        daily_factor = 0.7 + 0.3 * np.sin((hour - 6) * np.pi / 12) if 6 <= hour <= 22 else 0.6
        
        for _, eq in equipment_df.sample(frac=0.1).iterrows():  # Sample 10% each interval
            eq_type = eq['equipment_type']
            base = base_values.get(eq_type, base_values['Transformer'])
            
            # Check for anomaly
            is_anomaly = random.random() < ANOMALY_PROBABILITY
            anomaly_type = None
            
            if is_anomaly:
                anomaly_type = random.choice(['overvoltage', 'undervoltage', 'overcurrent', 'frequency_deviation'])
            
            # Generate measurements with noise
            voltage_pu = base['voltage'] * daily_factor + np.random.normal(0, 0.02)
            current_pu = base['current'] * daily_factor + np.random.normal(0, 0.05)
            power_mw = base['power'] * daily_factor * eq['rated_voltage_kv'] + np.random.normal(0, 5)
            frequency_hz = base['frequency'] + np.random.normal(0, 0.01)
            
            # Apply anomaly effects
            if anomaly_type == 'overvoltage':
                voltage_pu *= random.uniform(1.1, 1.2)
            elif anomaly_type == 'undervoltage':
                voltage_pu *= random.uniform(0.8, 0.9)
            elif anomaly_type == 'overcurrent':
                current_pu *= random.uniform(1.3, 1.8)
            elif anomaly_type == 'frequency_deviation':
                frequency_hz += random.choice([-0.5, 0.5])
            
            readings.append({
                'reading_id': str(uuid.uuid4()),
                'equipment_id': eq['equipment_id'],
                'substation_id': eq['substation_id'],
                'timestamp': current_time,
                'voltage_pu': round(voltage_pu, 4),
                'current_pu': round(current_pu, 4),
                'active_power_mw': round(power_mw, 2),
                'reactive_power_mvar': round(power_mw * random.uniform(0.2, 0.4), 2),
                'frequency_hz': round(frequency_hz, 3),
                'is_anomaly': is_anomaly,
                'anomaly_type': anomaly_type
            })
        
        current_time += interval
    
    return pd.DataFrame(readings)

# Generate 1 hour of SCADA data for demonstration
demo_start = START_DATE
demo_end = START_DATE + timedelta(hours=1)
df_scada = generate_scada_readings(df_equipment, demo_start, demo_end, SCADA_READINGS_PER_MINUTE)
print(f"Generated {len(df_scada)} SCADA readings")
print(f"Anomalies detected: {df_scada['is_anomaly'].sum()}")
df_scada.head()

## 3. Generate PMU (Synchrophasor) Data

In [ ]:
def generate_pmu_readings(substations_df, start_time, duration_seconds, samples_per_second=30):
    """Generate high-frequency PMU synchrophasor data"""
    
    readings = []
    pmu_substations = substations_df[substations_df['is_critical']].copy()
    
    for _, sub in pmu_substations.iterrows():
        base_angle = random.uniform(-30, 30)  # degrees
        base_frequency = 60.0
        
        for i in range(duration_seconds * samples_per_second):
            timestamp = start_time + timedelta(seconds=i/samples_per_second)
            
            # Add realistic oscillations
            time_factor = i / samples_per_second
            angle_variation = 2 * np.sin(2 * np.pi * 0.1 * time_factor)  # 0.1 Hz oscillation
            
            readings.append({
                'pmu_id': f'PMU-{sub["substation_id"]}',
                'substation_id': sub['substation_id'],
                'timestamp': timestamp,
                'voltage_magnitude_pu': round(1.0 + np.random.normal(0, 0.005), 5),
                'voltage_angle_deg': round(base_angle + angle_variation + np.random.normal(0, 0.1), 3),
                'current_magnitude_pu': round(0.7 + np.random.normal(0, 0.02), 5),
                'current_angle_deg': round(base_angle - 25 + np.random.normal(0, 0.1), 3),
                'frequency_hz': round(base_frequency + np.random.normal(0, 0.005), 4),
                'rocof_hz_s': round(np.random.normal(0, 0.01), 5)  # Rate of change of frequency
            })
    
    return pd.DataFrame(readings)

# Generate 10 seconds of PMU data (for demonstration)
df_pmu = generate_pmu_readings(df_substations, START_DATE, duration_seconds=10)
print(f"Generated {len(df_pmu)} PMU readings")
df_pmu.head()

## 4. Generate Weather Data

In [ ]:
def generate_weather_data(substations_df, start_date, end_date):
    """Generate hourly weather data for each substation"""
    
    readings = []
    current_time = start_date
    
    while current_time < end_date:
        hour = current_time.hour
        day_of_year = current_time.timetuple().tm_yday
        
        for _, sub in substations_df.iterrows():
            # Base temperature varies by region and time
            region_temp = {'Northeast': 40, 'Southeast': 60, 'Midwest': 45, 'Southwest': 70, 'West': 55}
            base_temp = region_temp.get(sub['region'], 50)
            
            # Add seasonal and daily variation
            seasonal_var = 20 * np.cos(2 * np.pi * (day_of_year - 172) / 365)
            daily_var = 10 * np.sin((hour - 6) * np.pi / 12) if 6 <= hour <= 18 else -5
            
            temperature_f = base_temp + seasonal_var + daily_var + np.random.normal(0, 3)
            
            readings.append({
                'substation_id': sub['substation_id'],
                'timestamp': current_time,
                'temperature_f': round(temperature_f, 1),
                'humidity_pct': round(max(20, min(100, 60 + np.random.normal(0, 15))), 1),
                'wind_speed_mph': round(max(0, 8 + np.random.normal(0, 5)), 1),
                'precipitation_in': round(max(0, np.random.exponential(0.02)), 3),
                'solar_radiation_wm2': round(max(0, 500 * np.sin((hour - 6) * np.pi / 12)), 1) if 6 <= hour <= 18 else 0
            })
        
        current_time += timedelta(hours=1)
    
    return pd.DataFrame(readings)

# Generate 1 day of weather data
df_weather = generate_weather_data(df_substations, START_DATE, START_DATE + timedelta(days=1))
print(f"Generated {len(df_weather)} weather records")
df_weather.head()

## 5. Generate Anomaly Events Summary

In [ ]:
def generate_anomaly_events(scada_df):
    """Aggregate anomalies into event records"""
    
    anomalies = scada_df[scada_df['is_anomaly']].copy()
    
    if len(anomalies) == 0:
        return pd.DataFrame()
    
    severity_map = {
        'overvoltage': random.choice(['Warning', 'Critical']),
        'undervoltage': random.choice(['Warning', 'Critical']),
        'overcurrent': 'Critical',
        'frequency_deviation': random.choice(['Info', 'Warning'])
    }
    
    events = []
    for _, row in anomalies.iterrows():
        events.append({
            'event_id': str(uuid.uuid4()),
            'equipment_id': row['equipment_id'],
            'substation_id': row['substation_id'],
            'event_timestamp': row['timestamp'],
            'event_type': row['anomaly_type'],
            'severity': severity_map.get(row['anomaly_type'], 'Info'),
            'voltage_reading': row['voltage_pu'],
            'current_reading': row['current_pu'],
            'frequency_reading': row['frequency_hz'],
            'acknowledged': random.random() < 0.7,
            'resolved': random.random() < 0.5
        })
    
    return pd.DataFrame(events)

df_anomaly_events = generate_anomaly_events(df_scada)
print(f"Generated {len(df_anomaly_events)} anomaly events")
if len(df_anomaly_events) > 0:
    display(df_anomaly_events.head())

## 6. Data Validation

In [ ]:
# Validate data quality
print("=== Data Validation Report ===")
print(f"\nSubstations: {len(df_substations)} records")
print(f"  - Null values: {df_substations.isnull().sum().sum()}")
print(f"  - Unique IDs: {df_substations['substation_id'].nunique()}")

print(f"\nEquipment: {len(df_equipment)} records")
print(f"  - Null values: {df_equipment.isnull().sum().sum()}")
print(f"  - Valid FK references: {df_equipment['substation_id'].isin(df_substations['substation_id']).all()}")

print(f"\nSCADA Readings: {len(df_scada)} records")
print(f"  - Date range: {df_scada['timestamp'].min()} to {df_scada['timestamp'].max()}")
print(f"  - Anomaly rate: {df_scada['is_anomaly'].mean()*100:.2f}%")

print(f"\nPMU Readings: {len(df_pmu)} records")
print(f"  - Frequency range: {df_pmu['frequency_hz'].min():.4f} - {df_pmu['frequency_hz'].max():.4f} Hz")

print(f"\nWeather: {len(df_weather)} records")
print(f"  - Temperature range: {df_weather['temperature_f'].min():.1f} - {df_weather['temperature_f'].max():.1f} °F")

## 7. Write to Lakehouse

In [ ]:
# Convert to Spark DataFrames and write to Lakehouse
# Uncomment the following code when running in Microsoft Fabric

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()

# # Write dimension tables (overwrite mode)
# spark.createDataFrame(df_substations).write.mode("overwrite").format("delta").saveAsTable("dim_substations")
# spark.createDataFrame(df_equipment).write.mode("overwrite").format("delta").saveAsTable("dim_equipment")

# # Write fact tables (append mode for streaming)
# spark.createDataFrame(df_scada).write.mode("append").format("delta").saveAsTable("fact_scada_measurements")
# spark.createDataFrame(df_pmu).write.mode("append").format("delta").saveAsTable("fact_pmu_readings")
# spark.createDataFrame(df_weather).write.mode("append").format("delta").saveAsTable("fact_weather")
# spark.createDataFrame(df_anomaly_events).write.mode("append").format("delta").saveAsTable("fact_anomaly_events")

# print("Data written to Lakehouse successfully!")

# For local testing, save as CSV
df_substations.to_csv('dim_substations.csv', index=False)
df_equipment.to_csv('dim_equipment.csv', index=False)
df_scada.to_csv('fact_scada_measurements.csv', index=False)
print("Data saved to CSV for local testing")

## 8. Sample Visualizations

In [ ]:
import matplotlib.pyplot as plt

# Plot voltage readings over time
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Voltage distribution
axes[0, 0].hist(df_scada['voltage_pu'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Voltage (p.u.)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Voltage Distribution')
axes[0, 0].axvline(x=1.0, color='r', linestyle='--', label='Nominal')
axes[0, 0].legend()

# Frequency readings
axes[0, 1].hist(df_pmu['frequency_hz'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_xlabel('Frequency (Hz)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('PMU Frequency Distribution')
axes[0, 1].axvline(x=60.0, color='r', linestyle='--', label='Nominal')
axes[0, 1].legend()

# Anomaly types
anomaly_counts = df_scada[df_scada['is_anomaly']]['anomaly_type'].value_counts()
if len(anomaly_counts) > 0:
    axes[1, 0].bar(anomaly_counts.index, anomaly_counts.values, color='coral')
    axes[1, 0].set_xlabel('Anomaly Type')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Anomaly Distribution')
    axes[1, 0].tick_params(axis='x', rotation=45)

# Equipment by type
eq_counts = df_equipment['equipment_type'].value_counts()
axes[1, 1].pie(eq_counts.values, labels=eq_counts.index, autopct='%1.1f%%')
axes[1, 1].set_title('Equipment by Type')

plt.tight_layout()
plt.show()

## Known Limitations

1. **Simplified Anomaly Model**: Real grid anomalies have complex temporal patterns and correlations
2. **No Geographic Correlation**: Weather patterns should be correlated across nearby substations
3. **Fixed Topology**: Real grids have dynamic switching configurations
4. **Synthetic IDs**: Equipment IDs follow a simple pattern rather than real naming conventions

## Next Steps

1. Connect this data to Eventstream for real-time ingestion
2. Create KQL queries in Eventhouse for anomaly detection
3. Build Real-Time Dashboard for grid monitoring
4. Configure Activator alerts for critical anomalies